# FP8 vs FP16 vs SAEBench — feature comparison

Analyzes the output of `experiments/compare_features.py`, which compares the
**learned decoder features** of three BatchTopK/TopK SAE families that all live in
the same residual stream, width 65,536, at the **100M-token** checkpoint. Run for
two base models (set `MODEL` in the config cell):

* **gemma-2-2b** — `blocks.12.hook_resid_post` (SAEBench: `canrager/saebench_gemma-2-2b_width-2pow16_date-1109`)
* **pythia-160m-deduped** — `blocks.8.hook_resid_post` (SAEBench: `adamkarvonen/saebench_pythia-160m-deduped_width-2pow16_date-0108`, TopK family)

For each model the three SAE families are:

* **FP16** — our standard BatchTopK k-sweep (`results/saebench_<model>`)
* **FP8** — our fp8 BatchTopK k-sweep (`results/saebench_<model>_fp8`)
* **SAEBench** — the published authors' TopK SAEs

Our k grid is `{50, 100, 250, 500, 1000, 2500}`; the SAEBench trainers are
`k={20,40,80,160,320,640}`, so each of our k is paired to the **nearest-k** SAEBench
trainer (k=1000/2500 only map to 640 — treat those as "nearest available"). FP16↔FP8
are always step- and k-matched (both at 100M tokens).

### Two analyses

**Part A — same concept, same feature?** For each SAEBench *sparse-probing* concept
class we pick the single most discriminative SAE feature (top-1, by
`|mean_pos − mean_neg|`, exactly as sparse probing does) and cosine-compare the
SAEs' top-1 features. We also report the **best-match** cosine of one SAE's top-1
feature against the *entire* other dictionary, to distinguish "different direction"
from "just not the other SAE's #1".

**Part B — is every feature mirrored?** For every feature in SAE A we find its
closest counterpart in SAE B (max cosine over all of B), both directions, per pair.
The distribution measures how completely one dictionary reproduces another.

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

FC_ROOT = Path("/wekafs/smerrill/efficient_sae/experiments/results/feature_comparison")
MODEL = "gemma-2-2b"          # <- switch to "pythia-160m-deduped" for the pythia run
RESULTS_ROOT = FC_ROOT / MODEL
print("available model runs:", [p.name for p in FC_ROOT.iterdir() if p.is_dir()])
ROLES = ["fp16", "fp8", "saebench"]
PAIRS = ["fp16-fp8", "fp16-saebench", "fp8-saebench"]          # Part A cosine keys
PAIRS_US = ["fp16__fp8", "fp16__saebench", "fp8__saebench"]    # Part B npz keys
PAIR_COLORS = {"fp16-fp8": "#1f77b4", "fp16-saebench": "#d62728", "fp8-saebench": "#2ca02c"}

summary = json.loads((RESULTS_ROOT / "summary.json").read_text())
KS = sorted(int(k) for k in summary["k_results"])
print("model:", summary["model"], "| hook:", summary["hook"], "| width:", summary["width"])
print("k-sweep present:", KS)
print("\nk pairing (our k -> SAEBench trainer):")
for k in KS:
    p = summary["k_pairing"][str(k)]
    flag = "" if p["fp16_step"] == p["fp8_step"] else "  [steps NOT matched]"
    print(f"  k={k:5d}  fp16/fp8 step {p['fp16_step']}  ->  SAEBench trainer_{p['saebench_trainer']} (k={p['saebench_k']}){flag}")

available model runs: ['pythia-160m-deduped', 'gemma-2-2b']


FileNotFoundError: [Errno 2] No such file or directory: '/wekafs/smerrill/efficient_sae/experiments/results/feature_comparison/gemma-2-2b/summary.json'

## Part A — top-1 probing feature per concept

For every concept class (8 datasets, ~36 classes total) we found each SAE's single
most discriminative feature. Below:

1. **Headline** — mean cosine between the SAEs' *own* top-1 features, per k.
2. **Best-match** — mean cosine of one SAE's top-1 feature against the *whole* other
   dictionary (how recoverable the concept direction is at all), per k.
3. Per-concept detail table for a chosen k.

In [ ]:
# Load every per-k Part A result into one tidy DataFrame (one row per concept per k).
rows = []
for k in KS:
    f = RESULTS_ROOT / f"k{k}" / "topk_cosine.json"
    if not f.exists():
        continue
    blob = json.loads(f.read_text())
    for c in blob["concepts"]:
        rec = {"k": k, "dataset": c["dataset"].split("/")[-1], "class": c["class"]}
        for pr in PAIRS:
            rec[f"cos_{pr}"] = c["cos"].get(pr, np.nan)
            rec[f"abscos_{pr}"] = c["abscos"].get(pr, np.nan)
        for d, bm in c["bestmatch"].items():
            rec[f"best_{d}"] = bm["cos"]
        for role in ROLES:
            rec[f"top1_{role}"] = c["top1"].get(role)
        rows.append(rec)
partA = pd.DataFrame(rows)
print(f"{len(partA)} concept rows across {partA['k'].nunique()} k values "
      f"({partA['dataset'].nunique()} datasets, {partA.groupby('k').size().iloc[0]} concepts/k)")
partA.head()

In [ ]:
# Headline (top1-vs-top1) and best-match cosine vs k.
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
for pr in PAIRS:
    g = partA.groupby("k")[f"abscos_{pr}"].mean()
    ax.plot(g.index, g.values, "o-", color=PAIR_COLORS[pr], label=pr)
ax.set(xscale="log", xlabel="our k (BatchTopK)", ylabel="mean |cosine|",
       title="Part A: top-1 vs top-1\n(do the SAEs pick the same #1 feature direction?)")
ax.set_xticks(KS); ax.set_xticklabels(KS); ax.grid(alpha=.3); ax.legend()

ax = axes[1]
bm_dirs = [c for c in partA.columns if c.startswith("best_")]
for d in bm_dirs:
    g = partA.groupby("k")[d].mean()
    style = "--" if d.split("best_")[1].startswith("saebench") else "-"
    ax.plot(g.index, g.values, "o" + style, label=d.replace("best_", ""), alpha=.85)
ax.set(xscale="log", xlabel="our k (BatchTopK)", ylabel="mean best-match cosine",
       title="Part A: top-1 feature's best match in the other dictionary\n"
             "(is the concept direction present at all?)")
ax.set_xticks(KS); ax.set_xticklabels(KS); ax.grid(alpha=.3); ax.legend(fontsize=8, ncol=2)
plt.tight_layout(); plt.show()

In [ ]:
# Per-concept detail for one k: top-1 feature index in each SAE + pairwise cosines.
K_DETAIL = 500  # <- change to inspect another k

d = partA[partA.k == K_DETAIL].copy()
show = d[["dataset", "class",
          "top1_fp16", "top1_fp8", "top1_saebench",
          "cos_fp16-fp8", "cos_fp16-saebench", "cos_fp8-saebench",
          "best_fp16->fp8", "best_fp8->fp16"]]
show = show.rename(columns={"cos_fp16-fp8": "cos(16,8)",
                            "cos_fp16-saebench": "cos(16,sb)",
                            "cos_fp8-saebench": "cos(8,sb)",
                            "best_fp16->fp8": "best 16->8",
                            "best_fp8->fp16": "best 8->16"})
print(f"k = {K_DETAIL}: top-1 feature per concept and matched cosines")
show.round(3).reset_index(drop=True)

## Part B — every feature's closest counterpart

For each feature in SAE A, its **max cosine** to any feature in SAE B (both
directions, every pair, every k). High mass near 1.0 ⇒ the two dictionaries learn
largely the same directions; mass near 0 ⇒ many features in A have no analog in B.

In [ ]:
# Load Part B per-feature max-cosine arrays for every k.
def load_partB(k):
    f = RESULTS_ROOT / f"k{k}" / "allfeat_match.npz"
    return dict(np.load(f)) if f.exists() else None

partB = {k: load_partB(k) for k in KS if load_partB(k) is not None}

# Summary line: median best-match cosine vs k (averaging the two directions per pair).
fig, ax = plt.subplots(figsize=(8, 5))
for pr in PAIRS_US:
    label = pr.replace("__", "-")
    med = []
    for k in partB:
        v = np.concatenate([partB[k][f"{pr}__A2B_maxcos"], partB[k][f"{pr}__B2A_maxcos"]])
        med.append(np.median(v))
    ax.plot(list(partB.keys()), med, "o-", color=PAIR_COLORS[label], label=label)
ax.set(xscale="log", xlabel="our k (BatchTopK)", ylabel="median best-match cosine",
       title="Part B: median nearest-feature cosine vs k (both directions pooled)")
ax.set_xticks(list(partB.keys())); ax.set_xticklabels(list(partB.keys()))
ax.grid(alpha=.3); ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
# Distribution of per-feature best-match cosine, one panel per k (A->B direction).
ks_plot = list(partB.keys())
fig, axes = plt.subplots(1, len(ks_plot), figsize=(3.2 * len(ks_plot), 3.4), sharey=True)
if len(ks_plot) == 1:
    axes = [axes]
bins = np.linspace(0, 1, 41)
for ax, k in zip(axes, ks_plot):
    for pr in PAIRS_US:
        label = pr.replace("__", "-")
        v = partB[k][f"{pr}__A2B_maxcos"]
        ax.hist(v, bins=bins, histtype="step", color=PAIR_COLORS[label],
                label=label, density=True, linewidth=1.5)
    ax.set(title=f"k = {k}", xlabel="best-match cosine")
    ax.grid(alpha=.3)
axes[0].set_ylabel("density")
axes[-1].legend(fontsize=7)
fig.suptitle("Part B: per-feature nearest-cosine distribution (A→B)", y=1.04)
plt.tight_layout(); plt.show()

In [ ]:
# Part B summary table: mean / median / fraction>0.9 for each pair & direction & k.
brows = []
for k in partB:
    summ = json.loads((RESULTS_ROOT / f"k{k}" / "allfeat_match_summary.json").read_text())
    for pr, s in summ.items():
        brows.append({
            "k": k, "pair": pr.replace("__", "-"),
            "A2B_mean": s["A2B_mean"], "A2B_median": s["A2B_median"],
            "B2A_mean": s["B2A_mean"], "B2A_median": s["B2A_median"],
            "A2B_frac>0.9": s["A2B_frac_gt_0.9"], "B2A_frac>0.9": s["B2A_frac_gt_0.9"],
        })
partB_summary = pd.DataFrame(brows).sort_values(["pair", "k"]).reset_index(drop=True)
partB_summary.round(3)

## How to read this

* **Part A top-1-vs-top-1** is a *strict* test: it asks whether two SAEs independently
  nominate the *same direction* as the single most useful feature for a concept. It is
  often low even for very similar SAEs, because many near-equivalent features can be the
  argmax.
* **Part A best-match** is the fair "is the concept direction there?" test. FP16↔FP8
  best-match well above the FP16/FP8↔SAEBench best-match means FP8 reproduces our own
  FP16 dictionary far more faithfully than the (differently-trained, nearest-k)
  published SAEBench SAEs reproduce either.
* **Part B** generalizes this to the whole dictionary. Compare the FP16↔FP8 curve to the
  ↔SAEBench curves: the gap is the "precision cost" of FP8 relative to the much larger
  difference introduced by a different training run / recipe.
* **k caveat:** SAEBench tops out at k=640, so for our k=1000 and k=2500 the SAEBench
  comparison is against a sparser SAE (nearest available), which inflates the apparent
  FP16/FP8-vs-SAEBench difference. FP16↔FP8 is always step- and k-matched.

To regenerate results: `python experiments/compare_features.py --parts all`.

## Part B+ — bijective hardening (is the greedy match honest?)

Part B's nearest-cosine is **greedy**: many features in A can map onto the same
feature in B, which *overstates* similarity. These diagnostics (from
`allfeat_match_summary.json`) quantify how far greedy is from a true 1-1 matching:

* **greedy vs bijective median** — if the solid (1-1) line tracks the dashed (greedy)
  line, greedy wasn't cheating; the dictionary overlap is real.
* **collision rate** — fraction of A-features that had to share a B-target. Low ⇒
  greedy ≈ bijective.
* **mutual-match rate** — fraction of mutual nearest neighbours (A→B→A returns to
  start); the permutation-honest "these are genuinely paired" number.
* **self-match rate** (printed in the table) — fraction with `argmax(i)==i`. For
  FP16↔FP8 this is ~0: same seed/data, but the FP8 GEMMs perturb the optimization
  enough that features end up **permuted**, not index-aligned. That's *why* every
  functional comparison below matches features via cosine rather than by index.

In [ ]:
from IPython.display import display

# Load the bijective/collision diagnostics added to Part B.
bij_rows = []
for k in KS:
    f = RESULTS_ROOT / f"k{k}" / "allfeat_match_summary.json"
    if not f.exists():
        continue
    for pr, s in json.loads(f.read_text()).items():
        if "greedy_bijective_median" not in s:
            continue  # old result without bijective stats — rerun compare_features.py
        bij_rows.append({
            "k": k, "pair": pr.replace("__", "-"),
            "greedy_median": s["A2B_median"],
            "bijective_median": s["greedy_bijective_median"],
            "bijective_frac>0.9": s["greedy_bijective_frac_gt_0.9"],
            "coverage": s["greedy_bijective_coverage"],
            "mutual_rate": s["mutual_rate"],
            "self_match_rate": s["self_match_rate"],
            "collision_rate": s["collision_rate"],
            "max_mult": s["max_multiplicity"],
        })

if not bij_rows:
    print("No bijective stats found — rerun: python experiments/compare_features.py "
          "--parts b --force")
else:
    bij = pd.DataFrame(bij_rows).sort_values(["pair", "k"]).reset_index(drop=True)
    display(bij.round(3))

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    ax = axes[0]
    for pr in PAIRS:
        g = bij[bij.pair == pr]
        if g.empty:
            continue
        ax.plot(g.k, g.greedy_median, "o--", color=PAIR_COLORS[pr], alpha=.45,
                label=f"{pr} greedy")
        ax.plot(g.k, g.bijective_median, "o-", color=PAIR_COLORS[pr],
                label=f"{pr} bijective")
    ax.set(xscale="log", xlabel="our k (BatchTopK)", ylabel="median matched cosine",
           title="Greedy (dashed) vs true 1-1 bijective (solid) matched cosine\n"
                 "(overlap ⇒ greedy wasn't inflating similarity)")
    ax.set_xticks(KS); ax.set_xticklabels(KS); ax.grid(alpha=.3); ax.legend(fontsize=7)

    ax = axes[1]
    for pr in PAIRS:
        g = bij[bij.pair == pr]
        if g.empty:
            continue
        ax.plot(g.k, g.collision_rate, "o-", color=PAIR_COLORS[pr],
                label=f"{pr} collision")
        ax.plot(g.k, g.mutual_rate, "s--", color=PAIR_COLORS[pr], alpha=.6,
                label=f"{pr} mutual")
    ax.set(xscale="log", xlabel="our k (BatchTopK)", ylabel="rate",
           title="Collision rate (solid, lower=better) &\nmutual-match rate (dashed, higher=better)")
    ax.set_xticks(KS); ax.set_xticklabels(KS); ax.grid(alpha=.3); ax.legend(fontsize=7)
    plt.tight_layout(); plt.show()

## Part C — functional equivalence (do the features *act* the same?)

Parts A/B compare decoder **directions** (geometry). The obvious rebuttal is *"same
directions — but do they actually fire the same on real tokens?"* Part C answers that
by running every SAE on an **identical held-out token block** and measuring, on the
same inputs (from `functional_summary.json` / `functional.npz`):

* **recon cosine** (every pair) — cosine between the two SAEs' *reconstructions* of
  the **same activation**. Permutation-invariant and end-to-end: the most direct
  "they compute the same function" number. FP16↔FP8 near 1.0 ⇒ the 8-bit GEMMs don't
  change what the SAE does.
* **matched-feature activation correlation** (FP16↔FP8) — for each **bijectively
  matched** feature pair, the Pearson correlation of its activation across tokens.
  High ⇒ matched features don't just point the same way, they *fire* together.
* **active-set Jaccard** (FP16↔FP8) — per token, the overlap of *which* matched
  features are on. (Naturally lower at tiny k, where a handful of active features out
  of 65k makes the set overlap noisy — read it alongside recon cosine, not alone.)

Generate with `python experiments/compare_features.py --parts c` (use a larger
`--func-tokens`, e.g. 8192–16384, so more features fire enough times to score).

In [ ]:
# Part C summary across k (recon cosine for all pairs; matched-feature stats fp16<->fp8).
fc_rows = []
for k in KS:
    f = RESULTS_ROOT / f"k{k}" / "functional_summary.json"
    if not f.exists():
        continue
    for pr, s in json.loads(f.read_text()).items():
        fc_rows.append({"k": k, "pair": pr.replace("__", "-"), **s})

if not fc_rows:
    print("No Part C results — run: python experiments/compare_features.py --parts c "
          "--func-tokens 8192")
else:
    fc = pd.DataFrame(fc_rows).sort_values(["pair", "k"]).reset_index(drop=True)
    display(fc.round(4))

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    ax = axes[0]
    for pr in PAIRS:
        g = fc[fc.pair == pr].dropna(subset=["recon_cos_median"])
        if g.empty:
            continue
        ax.plot(g.k, g.recon_cos_median, "o-", color=PAIR_COLORS[pr], label=pr)
    ax.set(xscale="log", xlabel="our k (BatchTopK)", ylabel="median recon cosine",
           title="Part C: same-input reconstruction cosine\n"
                 "(do the SAEs map the same activation to the same output?)")
    ax.set_xticks(KS); ax.set_xticklabels(KS); ax.grid(alpha=.3); ax.legend()

    ax = axes[1]
    g = fc[fc.pair == "fp16-fp8"].dropna(subset=["feat_corr_median"])
    if not g.empty:
        ax.plot(g.k, g.feat_corr_median, "o-", color=PAIR_COLORS["fp16-fp8"],
                label="matched feat-corr (median)")
        ax.plot(g.k, g["feat_corr_frac_gt_0.9"], "^:", color=PAIR_COLORS["fp16-fp8"],
                alpha=.7, label="frac feat-corr > 0.9")
        ax.plot(g.k, g.active_jaccard_median, "s--", color="#9467bd",
                label="active-set Jaccard (median)")
    ax.set(xscale="log", xlabel="our k (BatchTopK)", ylabel="value",
           title="Part C: FP16↔FP8 matched-feature functional agreement")
    ax.set_xticks(KS); ax.set_xticklabels(KS); ax.grid(alpha=.3); ax.legend(fontsize=8)
    plt.tight_layout(); plt.show()

In [ ]:
# Part C distributions for the FP16<->FP8 matched pair, one line per k.
def load_func(k):
    f = RESULTS_ROOT / f"k{k}" / "functional.npz"
    return dict(np.load(f)) if f.exists() else None

funcB = {k: load_func(k) for k in KS if load_func(k) is not None}
key = "fp16__fp8"
if not funcB:
    print("No functional.npz yet — run compare_features.py --parts c.")
else:
    ks_plot = list(funcB.keys())
    panels = [("recon_cos", "recon cosine (per token)", (0, 1)),
              ("matched_feat_corr", "matched-feature activation corr", (-0.2, 1)),
              ("active_jaccard", "active-set Jaccard (per token)", (0, 1))]
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    for ax, (suffix, title, xlim) in zip(axes, panels):
        bins = np.linspace(*xlim, 41)
        for k in ks_plot:
            arr = funcB[k].get(f"{key}__{suffix}")
            if arr is None:
                continue
            v = arr[~np.isnan(arr)]
            ax.hist(v, bins=bins, histtype="step", density=True, linewidth=1.4,
                    label=f"k={k}")
        ax.set(title=title, xlabel="value"); ax.grid(alpha=.3); ax.legend(fontsize=7)
    axes[0].set_ylabel("density")
    fig.suptitle("Part C: FP16↔FP8 functional-equivalence distributions", y=1.03)
    plt.tight_layout(); plt.show()